# 03 · Scaling Up: Chunked Extraction and Resume

**Goal: pull months of data without timeouts or lost work.**

A one hour query is trivial. A one year query over a busy metric can return
millions of rows, exhaust your memory, and trip the Beehive's own time limits.
The professional pattern is to break a long window into chunks, pull each chunk,
and save progress so that a crash never costs more than a single chunk. That is
exactly the idea behind SageTools' `batchQuery`, and this notebook builds it up
from first principles so you understand every piece.

Notebook `02` is a prerequisite: this one assumes you are comfortable with the
`query` call and its time arguments.

In [ ]:
import sage_data_client
import pandas as pd
from datetime import datetime, timedelta
import os

In [ ]:
# Import the SageTools helper package (github.com/mpapka/SageTools).
#
# SageTools is a convenience layer built on top of the official sage_data_client.
# It is optional. Every lab in this course also shows the underlying
# sage_data_client call directly, so the helper is never a hard dependency.
#
# The try/except means the notebook runs whether or not the package is installed:
# if it is missing, haveSageTools becomes False and the notebook falls back to
# the direct calls it already teaches. To install it, see notebook 00.
try:
    import sage
    haveSageTools = True
except ImportError:
    haveSageTools = False

print("SageTools helper package available:", haveSageTools)

## 1. Why a single huge query fails

Two independent limits bite you as the window grows, and it helps to keep them
separate in your mind.

- **Server side timeout.** When you ask for a very large range, the Beehive has
  to gather and stream the whole result. Past some size it gives up and returns
  an error, often a 500 or a 504. This is not your code failing; it is the
  service protecting itself.
- **Client side memory.** Even if the server answered, a year of data sampled
  every few seconds can be tens of millions of rows. Loading that into a single
  pandas DataFrame can exhaust the memory on your machine.

Chunking solves both at once. Each chunk is small enough to answer quickly and to
hold in memory, and you write each chunk to disk before moving on, so your
program's memory use stays flat no matter how long the total window is.

## 2. Build a chunk schedule

Given a start, an end, and a chunk size, produce the list of `(chunkStart,
chunkEnd)` windows that tile the whole range. Everything downstream is just a
loop over this schedule.

Note the design: each chunk ends exactly where the next begins, and the final
chunk is clamped so it never runs past your requested end. That clamping matters,
because otherwise the last chunk would pull data you did not ask for.

In [ ]:
def buildChunks(startTime, endTime, chunkDays=7):
    """
    Tile a time range into a list of (chunkStart, chunkEnd) windows.

    Args:
        startTime, endTime: datetimes bounding the whole range.
        chunkDays: width of each chunk in days.

    Returns:
        A list of (chunkStart, chunkEnd) tuples. The final chunk is clamped to
        endTime so the schedule never overruns the requested range.
    """
    chunkDelta = timedelta(days=chunkDays)
    chunks = []
    cursor = startTime
    while cursor < endTime:
        chunkEnd = min(cursor + chunkDelta, endTime)
        chunks.append((cursor, chunkEnd))
        cursor = chunkEnd
    return chunks


windowStart = datetime(2024, 6, 1)
windowEnd = datetime(2024, 9, 1)
chunkSchedule = buildChunks(windowStart, windowEnd, chunkDays=7)

print(f"{len(chunkSchedule)} chunks of 7 days each")
for chunkStart, chunkEnd in chunkSchedule[:3]:
    print(f"  {chunkStart.date()} -> {chunkEnd.date()}")
print("  ...")
print(f"  last chunk ends {chunkSchedule[-1][1].date()} (equals window end: "
      f"{chunkSchedule[-1][1] == windowEnd})")

## 3. Pull each chunk and append to disk

The pattern writes each chunk to a growing CSV. The header is written once, with
the first chunk that returns data, and later chunks append rows only. A chunk that
returns nothing is skipped, not fatal, and a chunk that errors is caught so one
bad week does not sink the whole run.

This cell makes live queries. If you are offline, read it and skip to section 4,
which explains resuming without needing a network.

In [ ]:
def extractRange(nodeVsn, metricName, startTime, endTime, chunkDays=7,
                 outputCsv="extract.csv"):
    """
    Pull a long window in chunks, appending each chunk to one CSV file.

    Returns the total number of rows written.
    """
    chunks = buildChunks(startTime, endTime, chunkDays)
    isoFmt = "%Y-%m-%dT%H:%M:%SZ"
    wroteHeader = False
    totalRows = 0

    for chunkIdx, (chunkStart, chunkEnd) in enumerate(chunks, start=1):
        try:
            chunkDf = sage_data_client.query(
                start=chunkStart.strftime(isoFmt),
                end=chunkEnd.strftime(isoFmt),
                filter={"name": metricName, "vsn": nodeVsn},
            )
        except Exception as err:
            print(f"chunk {chunkIdx} failed ({err}), skipping")
            continue

        if chunkDf.empty:
            print(f"chunk {chunkIdx}/{len(chunks)} {chunkStart.date()}: empty")
            continue

        chunkDf.to_csv(outputCsv, mode="a", header=not wroteHeader, index=False)
        wroteHeader = True
        totalRows += len(chunkDf)
        print(f"chunk {chunkIdx}/{len(chunks)} {chunkStart.date()}: "
              f"{len(chunkDf)} rows (running total {totalRows})")

    print(f"done, {totalRows} rows in {outputCsv}")
    return totalRows


# Uncomment to run against the live Beehive:
# extractRange("W06C", "env.temperature", windowStart, windowEnd,
#              chunkDays=7, outputCsv="w06c_summer.csv")
print("extractRange defined")

## 4. Resume: never redo finished work

A three month pull can run for many minutes. If it crashes at chunk 40 of 100,
you do not want to restart at chunk 1. The fix is a **checkpoint**: after each
chunk completes, record how far you have gotten. On restart, skip any chunk whose
end is at or before the saved position.

The minimal version below stores a single high water mark timestamp in a small
text file. SageTools uses a richer pickle checkpoint that also survives a keyboard
interrupt, but the principle is identical.

Read the guard carefully. It skips a chunk when `chunkEnd <= resumePoint`. Using a
strict `<` instead would re-pull the boundary chunk, which is a classic off by one
bug worth understanding.

In [ ]:
def loadCheckpoint(checkpointFile):
    """Return the last completed chunkEnd as a datetime, or None if no file."""
    if os.path.exists(checkpointFile):
        with open(checkpointFile) as f:
            return datetime.fromisoformat(f.read().strip())
    return None


def saveCheckpoint(checkpointFile, chunkEnd):
    """Record the high water mark so a restart can skip finished chunks."""
    with open(checkpointFile, "w") as f:
        f.write(chunkEnd.isoformat())


def extractResumable(nodeVsn, metricName, startTime, endTime, chunkDays=7,
                     outputCsv="extract.csv", checkpointFile="extract.ckpt"):
    """Chunked extraction that resumes from a checkpoint after a crash."""
    resumePoint = loadCheckpoint(checkpointFile)
    if resumePoint:
        print(f"resuming, skipping everything up to {resumePoint}")

    chunks = buildChunks(startTime, endTime, chunkDays)
    isoFmt = "%Y-%m-%dT%H:%M:%SZ"
    wroteHeader = os.path.exists(outputCsv)

    for chunkStart, chunkEnd in chunks:
        if resumePoint and chunkEnd <= resumePoint:
            continue  # already done in a previous run

        chunkDf = sage_data_client.query(
            start=chunkStart.strftime(isoFmt),
            end=chunkEnd.strftime(isoFmt),
            filter={"name": metricName, "vsn": nodeVsn},
        )
        if not chunkDf.empty:
            chunkDf.to_csv(outputCsv, mode="a", header=not wroteHeader, index=False)
            wroteHeader = True

        saveCheckpoint(checkpointFile, chunkEnd)
        print(f"completed {chunkStart.date()} -> {chunkEnd.date()}")

    print("extraction complete")


print("extractResumable defined")

## 5. Choosing a chunk size

There is no single right chunk size. It is a tradeoff:

- **Smaller chunks** (hours) are safer against timeouts and use less memory, but
  they mean more requests, each with its own network overhead, so the total run
  is slower.
- **Larger chunks** (weeks) mean fewer requests and a faster run when things go
  well, but each request is more likely to time out, and a failure loses more
  work between checkpoints.

A good starting point is one week for a moderately busy metric, then adjust: if
you see timeouts, halve it; if every chunk returns instantly, double it. A robust
extractor can even do this automatically, shrinking the chunk on a failure and
retrying, which is one of the exercises below.

In [ ]:
# SageTools packages all of this, including adaptive chunk sizing, as a command
# line tool. Run it from a shell, not from the notebook:
if haveSageTools:
    print("python -m sage.beehive.extraction.batchQuery \\")
    print("    --name=env.temperature --vsn=W06C \\")
    print("    --start=2024-06-01 --chunkSize=7d")

## 6. Exercises

1. Change `buildChunks` to accept `chunkHours` instead of `chunkDays`. When is an
   hourly chunk the right call, and what does it cost you?
2. `extractResumable` skips a chunk when `chunkEnd <= resumePoint`. Rewrite the
   comparison as `<` and describe exactly which chunk would be pulled twice, and
   why that duplicates rows in the output CSV.
3. Extend the checkpoint to also store the running row count, so a restart can
   report how many rows were already collected before the crash.
   *Hint: write two lines to the checkpoint file, or store JSON.*
4. Make the extractor adaptive: when a chunk raises an exception, halve that
   chunk's width, split it into two sub windows, and retry each. Which failure
   does this fix, and which does it not?

## Further reading

- pandas `read_csv` and the `chunksize` argument: https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html
- The official data client: https://github.com/sagecontinuum/sage-data-client
- SageTools batch extraction: https://github.com/mpapka/SageTools